# Human agreement on the training videos

This notebook checks **how far the human coders agree with each other and with the model** on the training videos,
code by code, before they start on the validation sample. It follows the design of the human validation
([specification](../../docs/plans/human_validation/2026-10-08-validation-specification.md), sections 4 and 5):

- **Coder against coder** is the ceiling: if two trained people do not agree on a code, the model cannot be expected to.
- **Each coder against the model** and **the model's first pass against its second** sit beside it.
- Per code, every comparison is given as the validation table gives it (specification section 4, rewritten on
  2026-10-30): **raw agreement** first (the share of rows on which the two give the same answer; also **within one**
  point on a rating scale; within 5 seconds for the first decision), then **Krippendorff's alpha** (chance-corrected;
  nominal or ordinal by code; none for the first decision and the ask), **Gwet's AC1 or AC2** (chance-corrected in a
  way that stays readable when one answer dominates), alpha's **Landis & Koch band** (a descriptive name: slight,
  fair, moderate, substantial, almost perfect) and the row's **flags** (*prevalence*: one answer is at least 75% of
  the answers, so read AC1/AC2 rather than alpha; *small n*: fewer than 20 rows). The question concept is also shown
  through its **frame**, the row the validation table reports. Per field class (section B) the figures are alpha,
  stratified by item within a class, and plain agreement.
- The **bar** is alpha ≥ 0.61 by default, the lower bound of Landis & Koch's "substantial" band (read from
  `pipeline/codebooks/validation_reporting.yaml`, as `review_panel.py calibrate` reads it). It is a guide for the
  calibration discussion, not a gate: the validation table passes or fails nothing.

**One task page per coder** (`v3-dev9`; `review_panel.py training --clips`, the same page as the validation's
`session-export`, the researcher's "order A" of 2026-10-21). Each training video is one task: the coder watches the
**full video** (the copied file, played at the times the page shows) and answers, first, **seven presentation
cards** at the prepared presentation's window — the **three Stage A items** (positivity, enthusiasm, warmth), three
prepared-presentation codes (substance, coherence and the count of checkable claims, answered in bands) and the
ask — and then the **questions and answers**: the first-decision clock and **six codes for each drawn
question–answer pair** (the question's concept and severity; the answer's responsiveness, checkable support,
substance and coherence). There is no offer card and no topic card. As soon as both training videos are done, the
**comparisons** follow: which of the two pitches comes across as more positive, more enthusiastic, warmer (left or
right, with a "close" tick); on two videos this is their one pair shown three times, so the page has five tasks.
The comparisons come after the coder has watched both pitches' questioning (and usually the outcome), so they are
**not outcome-blind**. Every card that takes a value has a **"not sure"** tick beside the answer: the coder's best
guess is kept and the tables below also report it (section A counts the ticks). The page writes two kinds of sheet,
`clips__<coder>.csv` (the presentation cards and the comparisons) and `<video_id>__<coder>.csv` (the questions and
answers); the notebook reads both, a presentation card joins the same video's other cards by its path, and the
comparisons have their own section, H.

**How to use it.** Pick the project's `.venv` kernel, edit the configuration cell if needed, and run everything
(*Kernel → Restart & Run All*). It takes a few seconds.

**What it does not do.** It never changes the run: the coders' sheets are imported into a throwaway copy, exactly as
`review_panel.py calibrate` does, so the run's review logs and results stay untouched. With `EXPORT = False` it
writes nothing at all.

**How to read the numbers.** There are only **two training videos**, so each code has a handful of rows (a Stage A
item has two) and the comparisons are **one pair shown three times**. Alpha on so few rows swings widely and has no
interval (the video bootstrap needs at least five videos). Read the class rows and the plain agreement first, and use the disagreement list in section F to talk the
differences through; the per-code alphas are a guide to where to look, not a verdict.

In [ ]:
# ===== CONFIGURATION — the only cell you need to edit. Then: Kernel > Restart & Run All =====

RUN_NAME = "full-corpus-100"   # the run the training videos belong to: a folder under runs/ (or a full path)
RESULTS_DIR = None                  # the folder of the coders' filled training sheets (<video_id>__<coder>.csv
                                    #   and the session's clips__<coder>.csv: presentation cards and comparisons);
                                    #   None = runs/<run>/review/training/training_results
BAR = None                          # the alpha a field is read against; None = 0.61 (Landis & Koch "substantial")
FIGURE_WIDTH = "double"             # "double" (two journal columns) or "single", or a number of inches
EXPORT = False                      # False writes nothing. True saves the tables (CSV) and figures (PNG) under
                                    #   runs/<run>/review/training/agreement/

In [ ]:
# Setup: finds the project folder, imports the coders' sheets into a throwaway copy and pools them. Nothing to edit.
import sys
import warnings
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "analysis" / "human_agreement.py").exists())
sys.path.insert(0, str(REPO))
warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd
from IPython.display import display
from analysis import human_agreement as ha
from analysis import panel_explorer as px

pd.set_option("display.max_colwidth", 80)
RUN = px.resolve_run(RUN_NAME)
RESULTS = Path(RESULTS_DIR) if RESULTS_DIR else RUN / "review" / "training" / "training_results"
data = ha.load(RUN, RESULTS, bar=BAR)
print(f"Run: {RUN.name}\nSheets: {RESULTS.relative_to(REPO) if RESULTS.is_relative_to(REPO) else RESULTS}")
print(f"Coders: {', '.join(data.coders)}; training videos: {', '.join(data.videos)}; bar: alpha >= {data.bar:.2f}")
print("Comparisons: " + (f"on {', '.join(data.pairs)}" if data.pairs else "no session sheet, clips__<coder>.csv (section H is empty)"))
if data.import_failed:
    print("Some sheets could not be imported; the messages:\n" + data.import_log)

## A. The sheets

What each coder handed in: cards answered per video, how often they ticked **"I can't tell from the footage"** or
**"the unit is wrong"** (the card's turn, line of questioning or person is not what the page says), and how many cards
carry a note. A card with either box ticked counts in neither coder's answers below.

In [ ]:
sheets = ha.sheets_summary(data)
display(sheets)

## B. Agreement by field class

One row per kind of field. **Coder vs coder** is the ceiling; **reached the bar** compares its alpha with the bar.
The presentation cards' classes come first, in the order the task page asks them: Stage A (three items: its alpha
is computed within each item and pooled), the prepared presentation's content (three codes: substance, coherence
and the count of checkable claims, compared in its four bands; "wider context" is not on the coders' cards) and the
ask (judged by exact share in the paper table). Then the question and answer codes, which mix nominal codes
(concept, checkable support) with rating scales (severity, responsiveness, substance, coherence), so they have no
single alpha here: section C gives one per code. Then the first-decision clock, and last the comparisons. There is
no offer row and no topic row (neither is on the coders' page). "Rows" counts the cards both coders answered;
a best guess ticked "not sure" counts as an answer.

In [ ]:
by_class = ha.agreement_by_class(data)
display(by_class.drop(columns=["key", "scope", "class"], errors="ignore").round(2))

## C. Agreement by code

The same comparisons for each code (and each Stage A item), with the figures of the validation table for the code's
**kind** (categorical, binary, ordinal, clock, exact): for coder against coder, the coders (pooled) against the
model and the model's two passes, the **agreement** (identical; within 5 seconds for the first decision), **within
one** on a rating scale, **alpha** (none for the first decision and the ask), **AC1/AC2**, alpha's **band** and the
**flags**; each coder against the model keeps alpha and plain agreement. The question concept's row is followed by
its **frame**'s (promotion, prevention, unclassified, neutral), which is what the validation table reports for the
concept. On two training videos every row is flagged *small n*, and many carry *prevalence*. With two training
videos a Stage A item has **two rows**, so its alpha is close to meaningless: read Stage A from its class row above
and from the ratings in section D.

In [ ]:
by_code = ha.agreement_by_code(data)
display(by_code.drop(columns=["key", "scope"], errors="ignore").round(2))

In [ ]:
fig = ha.figure_agreement(data, by_code, width=FIGURE_WIDTH)
display(fig)
import matplotlib.pyplot as plt
plt.close(fig)

**Reading the figure.** Each row is a code; heavier lines separate the field classes. Black circles are the two
coders against each other, blue squares and orange triangles each coder against the model, hollow grey diamonds the
model against itself. Panel A is alpha with the bar as the dashed line; panel B the plain share of rows answered the
same way (within 5 seconds for the first decision, which has no alpha and so appears in panel B only). Where a coder sits far from both the other coder and the model, that coder is reading the code differently;
where both coders agree with each other but not with the model, the model is the odd one out.

## D. Stage A ratings, item by item

Each Stage A item's rating (1 to 5) on each training video, by each coder and by the model's first pass. The coders
rate the three items on the video's presentation cards, the first part of its task, while they watch the prepared
presentation at the times the page shows. On so few pitches this is more useful than an alpha: it shows which items
the coders read differently and by how much.

In [ ]:
fig = ha.figure_stage_a(data, width=FIGURE_WIDTH)
if fig is not None:
    display(fig)
    plt.close(fig)
display(ha.stage_a_ratings(data))

## E. Categorical codes: who chose what

For the codes with named values, how often each coder's answer met each of the other's. Agreement sits on the
diagonal; an off-diagonal cell is a pair of answers to talk through (section F lists the cards).

In [ ]:
for row in by_code[by_code["level"] == "nominal"].to_dict("records"):      # every code with named values
    table = ha.confusion(data, row["key"])
    if not table.empty:
        print(f"\n{row['code']}  ({row['key']})")
        display(table)

## F. The cards to talk through

Every card on which the coders gave different answers, or one of them could not decide, with the model's answers
(both passes) and both coders' notes. This is the list for the calibration session; the model-answers page
(`runs/<run>/review/training/model_answers.html`) shows the same cards with the footage and what the model recorded.

In [ ]:
talk = ha.disagreements(data)
print(f"{len(talk)} of {len(ha.cards(data))} cards")
display(talk)

## G. What to do next

- **Codes under the bar where the coders disagree with each other** (black circle low; on a row flagged
  *prevalence*, read AC1/AC2 and the plain agreement rather than alpha): the codebook entry is read
  in two ways. Go through its cards in section F with both coders, agree a reading, and add it to the codebook's
  examples if it is a recurring case.
- **One coder far from both the other and the model**: that coder needs a second look at the entry, not the codebook.
- **Both coders agree but the model does not**: a model weakness to record; it does not block the coders.
- Repeat on fresh development videos (`review_panel.py training --clips --from selection_all100.json --videos 2 --seed <new>`)
  until the researcher judges, from the bar, the plain agreement and the comparisons (section H), that the coders
  read the codes alike; then open the validation pages. The bar is a guide, not a gate.

## H. Comparisons: which of two pitches comes across as more …

The comparison tasks of the task page: for each pair of pitches, which one comes across as more positive, more
enthusiastic, warmer (left or right), with a "close" tick. A comparison is answered after the coder has done both
pitches' tasks, presentation and questioning alike, so it is not an outcome-blind judgement of the presentations.
The model answers every pair from its rank scores (`runs/<run>/ranking/scores.json`), so the coders' choices test
the scores with no model call. Per item: each coder's agreement with the model and the coders' pooled agreement,
coder against coder (the ceiling; chance is 0.50), how many "close" ticks fall on the narrowest third of the model's
score gaps, and each coder's coherence with their own 1-5 ratings of the same pitches. As in the validation table
there is **no target** (plan D, 2026-10-30): the coders' pooled agreement with the model, and coder against coder,
are given as k of n with an exact (Clopper–Pearson) 95% interval and a two-sided p **against 0.5**, a coin flip,
beside the number of "close" ticks; the binomial counts each coder's answer to a pair as one judgement, so with two
coders on the same pairs the interval and p are optimistic. On two training videos the three comparisons are the same two pitches shown three times
with their sides swapped by seed: a coder who chooses differently across them is reading the item differently from
one viewing to the next, which is what to talk through. The figure draws agreement by third of the score gap; with
one pair there is one gap, so read the table.

In [ ]:
pairs = ha.pairs_table(data)
if pairs.empty:
    print("No session sheet (clips__<coder>.csv) among the training sheets: no comparisons to show.")
else:
    display(pairs.round(2))
    fig = ha.figure_pairs(data, width=FIGURE_WIDTH)
    display(fig)
    plt.close(fig)

In [ ]:
if EXPORT:
    out = RUN / "review" / "training" / "agreement"
    out.mkdir(parents=True, exist_ok=True)
    for name, table in {"sheets": sheets, "by_class": by_class, "by_code": by_code, "stage_a": ha.stage_a_ratings(data),
                        "disagreements": talk, "cards": ha.cards(data), "pairs": pairs}.items():
        if not table.empty:
            table.to_csv(out / f"{name}.csv", index=False)
    for name, make in {"agreement_by_code": lambda: ha.figure_agreement(data, by_code, width=FIGURE_WIDTH),
                       "stage_a_ratings": lambda: ha.figure_stage_a(data, width=FIGURE_WIDTH),
                       "pairs": lambda: ha.figure_pairs(data, width=FIGURE_WIDTH)}.items():
        fig = make()
        if fig is not None:
            fig.savefig(out / f"{name}.png", dpi=300)
            plt.close(fig)
    print(f"Tables and figures written to {out.relative_to(REPO)}")
else:
    print("EXPORT = False: nothing was written.")